# Confirmatory chain runner: local prototype

Same structure as the cluster run: a model is served behind an OpenAI-compatible
endpoint by `serve_model.sh`, and the frozen `generate_chains_confirmatory.py`
(`--backend openai_compatible`) drives the chains. The notebook only holds the
parameters and the convenience steps; prompts, cache keys, seed derivation,
parsing, selection, and the edge-case policy all come from the frozen module.

**Start the server in a separate terminal first**:

```
# GPU workstation, tiny model first
./serve_model.sh --model HuggingFaceTB/SmolLM2-135M-Instruct

# 4080, Llama 8-bit (dev only; BF16 does not fit 16 GB)
./serve_model.sh --model meta-llama/Llama-3.1-8B-Instruct --quant bitsandbytes

# ASC-5, BF16 on two A100s
./serve_model.sh --model Qwen/Qwen3-32B --tp 2 --slurm
```

Prototyping runs on the GPU workstation, not on a CPU-only laptop. The serve
script picks `cpu` automatically only when no GPU is visible; GPU hosts keep
vLLM's default. Point `MODEL_ID` / `BASE_URL` below at the running server and
run the cells.

**Interrupt/resume**: the CLI writes partial chain logs and an append-only JSONL
response cache. Re-running the run cell skips completed hops and replays accepted
calls from cache. This is the frozen rerun semantics, not a notebook approximation.

Edit only the CONSTANTS cell between runs.

In [15]:
from pathlib import Path
import json
import os
import subprocess
import sys
import urllib.request

GENERATION_DIR = Path.cwd()
if not (GENERATION_DIR / "generate_chains_confirmatory.py").exists():
    # notebook started from the repo root instead of scripts/confirmatory
    GENERATION_DIR = GENERATION_DIR / "scripts" / "generation"
sys.path.insert(0, str(GENERATION_DIR))

import generate_chains_confirmatory as gcc  # for TASK_BANDS + path convention

GENERATOR = GENERATION_DIR / "generate_chains_confirmatory.py"
PYTHON = sys.executable
print("generation dir:", GENERATION_DIR)
print("python:", PYTHON)

generation dir: /home/user/code/echochains-vs-analysis/scripts/confirmatory
python: /home/user/miniconda3/envs/echochain/bin/python


In [22]:
# ---------------------------------------------------------------------
# CONSTANTS: the only cell to edit between runs
# ---------------------------------------------------------------------

# Must match --served-name of the running serve_model.sh process.
MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
# BASE_URL = "http://localhost:8558/v1"
BASE_URL = "http://a471-u099.psy.univie.ac.at:8892/v1"
API_KEY = "local"              # serve_model.sh default

TASK = "paraphrase"          # paraphrase | improve | take_inspiration | expand+compress
CONDITION = "vs_weighted"    # direct | uniform_list | vs_weighted

# Explicit decoding pair (works for models outside the pinned vendor table).
# For Llama-3.1-8B use 0.6 / 0.9 (the pinned card settings).
TEMPERATURE = 0.8
TOP_P = 0.9

# Chain budget (prototype values; the design uses 5 chains x 200 hops)
N_CHAINS = 1
N_HOPS = 3
LIMIT_SEEDS = 1              # None = all seeds in the seed list
LIMIT_CHAINS = None
LIMIT_HOPS = None
REPLICATE = 0

# Optional real seed file (schema: {"band": ..., "seeds": [...]}).
# None = write the toy seeds defined below.
SEEDS_FILE = None

PARTIAL_FLUSH_EVERY = 1      # flushes often so interrupts are cheap to test
MAX_RETRIES = 3

# Dev outputs live outside the real data tree.
DEV_ROOT = Path.home() / "echochains_dev"
LOGS_ROOT = DEV_ROOT / "logs"
CACHE_ROOT = DEV_ROOT / "cache"

print(json.dumps({"model": MODEL_ID, "base_url": BASE_URL, "task": TASK,
                  "condition": CONDITION, "chains": N_CHAINS, "hops": N_HOPS,
                  "temperature": TEMPERATURE, "top_p": TOP_P}, indent=2))

{
  "model": "HuggingFaceTB/SmolLM2-135M-Instruct",
  "base_url": "http://a471-u099.psy.univie.ac.at:8558/v1",
  "task": "paraphrase",
  "condition": "vs_weighted",
  "chains": 1,
  "hops": 3,
  "temperature": 0.8,
  "top_p": 0.9
}


In [23]:
# Server check: is serve_model.sh up and is MODEL_ID the name it exposes?
def list_served(base_url, api_key):
    req = urllib.request.Request(
        base_url.rstrip("/") + "/models",
        headers={"Authorization": f"Bearer {api_key}"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        return json.load(resp)

try:
    served = list_served(BASE_URL, API_KEY)
    names = [m.get("id") for m in served.get("data", [])]
    print("served:", names)
    assert MODEL_ID in names, (
        f"MODEL_ID {MODEL_ID!r} not served; start serve_model.sh with "
        f"--served-name {MODEL_ID} (or change MODEL_ID)")
except Exception as exc:
    raise RuntimeError(
        f"no server at {BASE_URL}; start serve_model.sh in another terminal") from exc

RuntimeError: no server at http://a471-u099.psy.univie.ac.at:8558/v1; start serve_model.sh in another terminal

In [ ]:
# Seeds: real file if given, otherwise write two toy sentence-band seeds.
BAND = gcc.TASK_BANDS[TASK]

if SEEDS_FILE is not None:
    BAND, SEEDS = gcc.load_seed_file(Path(SEEDS_FILE))
    assert BAND == gcc.TASK_BANDS[TASK], (
        f"seed band {BAND!r} != task band {gcc.TASK_BANDS[TASK]!r}")
    SEEDS_PATH = Path(SEEDS_FILE)
else:
    TOY = {
        "sentence": [
            "The committee postponed the vote until the budget office "
            "published its revised projections.",
            "A small crowd gathered at the harbor to watch the last of the "
            "fishing boats return before the storm.",
        ],
        "long": [
            "The committee postponed the vote until the budget office "
            "published its revised projections. Members said the delay was "
            "procedural rather than political, and that the revised numbers "
            "would determine whether the proposal reached the floor at all. "
            "Staff had worked through the weekend to reconcile the two sets "
            "of estimates, which differed by nearly a fifth of the total.",
        ],
    }
    SEEDS = [{"seed_id": i + 1, "text": t, "genre": "prototype"}
             for i, t in enumerate(TOY[BAND])]
    SEEDS_PATH = DEV_ROOT / f"toy_seeds_{BAND}.json"
    SEEDS_PATH.parent.mkdir(parents=True, exist_ok=True)
    SEEDS_PATH.write_text(json.dumps({"band": BAND, "seeds": SEEDS},
                                     ensure_ascii=False, indent=2),
                          encoding="utf-8")
    print(f"wrote {len(SEEDS)} toy {BAND}-band seed(s) to {SEEDS_PATH}")

for s in SEEDS:
    print(s["seed_id"], "|", s["text"][:70].replace("\n", " "), "...")

In [ ]:
# Progress scan: what is already on disk for this cell?
# Uses the module's path convention so it matches what the CLI writes.

def scan_progress(task, condition, model, seeds, chains, hops, logs_root):
    out_dir = gcc.cell_out_dir(logs_root, model, task, condition)
    rows = []
    for seed in seeds:
        for chain_id in range(1, chains + 1):
            base = out_dir / f"MSG_{seed['seed_id']:03d}_chain_{chain_id:03d}"
            done = 0
            for suffix in (".json", ".json.partial"):
                path = base.with_suffix(suffix)
                if path.exists():
                    log = json.loads(path.read_text(encoding="utf-8"))
                    done = max((e.get("hop", 0) for e in log), default=0)
                    break
            status = "complete" if done >= hops else ("partial" if done else "new")
            rows.append((seed["seed_id"], chain_id, done, status))
    return out_dir, rows

selected_seeds = SEEDS if LIMIT_SEEDS is None else SEEDS[:LIMIT_SEEDS]
out_dir, rows = scan_progress(TASK, CONDITION, MODEL_ID, selected_seeds,
                              N_CHAINS, N_HOPS, LOGS_ROOT)
done_total = sum(r[2] for r in rows)
print(f"cell dir: {out_dir}")
print(f"{done_total}/{len(rows) * N_HOPS} hops already on disk")
for seed_id, chain_id, done, status in rows:
    print(f"  seed {seed_id} chain {chain_id}: {done}/{N_HOPS} hops [{status}]")

In [ ]:
# Run / resume the cell through the frozen CLI. Safe to interrupt and re-run:
# completed hops are skipped, accepted calls come from the JSONL cache.

env = {
    "OPENAI_API_KEY": API_KEY,
    "OPENAI_BASE_URL": BASE_URL,
    "PATH": str(Path(PYTHON).parent) + ":" + os.environ.get("PATH", ""),
    "HOME": os.environ.get("HOME", ""),
}

cmd = [PYTHON, str(GENERATOR),
       "--backend", "openai_compatible",
       "--model", MODEL_ID,
       "--task", TASK,
       "--condition", CONDITION,
       "--seeds-file", str(SEEDS_PATH),
       "--chains", str(N_CHAINS),
       "--hops", str(N_HOPS),
       "--replicate", str(REPLICATE),
       "--partial-flush-every", str(PARTIAL_FLUSH_EVERY),
       "--max-retries", str(MAX_RETRIES),
       "--temperature", str(TEMPERATURE),
       "--top-p", str(TOP_P),
       "--logs-root", str(LOGS_ROOT),
       "--cache-root", str(CACHE_ROOT)]
if LIMIT_SEEDS is not None:
    cmd += ["--limit-seeds", str(LIMIT_SEEDS)]
if LIMIT_CHAINS is not None:
    cmd += ["--limit-chains", str(LIMIT_CHAINS)]
if LIMIT_HOPS is not None:
    cmd += ["--limit-hops", str(LIMIT_HOPS)]

print(" ".join(cmd), "\n")
result = subprocess.run(cmd, env=env, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"generator exited with {result.returncode}")

In [ ]:
# Re-scan and show the produced chain.
out_dir, rows = scan_progress(TASK, CONDITION, MODEL_ID, selected_seeds,
                              N_CHAINS, N_HOPS, LOGS_ROOT)
print(rows)

path = out_dir / f"MSG_{selected_seeds[0]['seed_id']:03d}_chain_001.json"
if path.exists():
    log = json.loads(path.read_text(encoding="utf-8"))
    for entry in log[:2] + log[-1:]:
        print(f"\nhop {entry['hop']} | seed {entry['random_seed']} "
              f"| from_cache={entry.get('from_cache')}")
        print(entry["text"][:400])
else:
    print("no completed chain file yet")